# CSA0812 – Assignment 3 | Q7: Contact-Merging Program
**Name:** Darshan A  
**Register Number:** 1973260011

## Stage 1 – Interpret the problem
- **Input:** contact records from two phones and a SIM card. Each record may contain a name, phone number and e-mail address.
- **Output:** one merged record for each identified contact, containing all available phone numbers and e-mail addresses.
- **Identity rule:** records with the same normalized phone number are treated as the same contact. A name match alone is not enough.
- **Data structures:** dictionary keyed by normalized phone number; sets for unique numbers, e-mails and name variants; lists for input and final output.
- **Measurable requirements:** equivalent phone formats produce the same key; duplicate records merge; unique e-mails and numbers are retained; missing fields do not crash the program; same-name contacts with different numbers stay separate.

## Stage 2 – Design
Phone normalization keeps digits only, removes a leading `91` from a 12-digit number, and removes a leading `0` from an 11-digit number. E-mails are stripped and lower-cased. Names are stripped and retained as variants. Contacts without a phone are kept separately rather than guessed to be duplicates.

### Pseudocode
1. Create an empty dictionary.
2. For each contact, normalize its phone number.
3. Use the normalized number as the key. If no number exists, create a unique key for that record.
4. If the key is new, create an entry with name, number set and e-mail set.
5. Add the original phone format and normalized number to the number set.
6. Add a non-empty lower-case e-mail to the e-mail set.
7. Retain the name spelling as a name variant.
8. Return the dictionary entries as a list.


In [9]:
def normalize_phone(phone):
    """Convert common Indian phone formats to a comparable digit key."""
    if not phone:
        return ""
    digits = "".join(ch for ch in str(phone) if ch.isdigit())
    if len(digits) == 12 and digits.startswith("91"):
        digits = digits[2:]
    elif len(digits) == 11 and digits.startswith("0"):
        digits = digits[1:]
    return digits


def merge_contacts(contacts):
    merged = {}
    for position, contact in enumerate(contacts):
        number_key = normalize_phone(contact.get("phone", ""))
        # Do not merge records with missing numbers based only on their names.
        key = number_key if number_key else f"NO_PHONE_{position}"

        if key not in merged:
            merged[key] = {
                "name": contact.get("name", "").strip() or "Unknown",
                "name_variants": set(),
                "numbers": set(),
                "emails": set()
            }

        entry = merged[key]
        name = contact.get("name", "").strip()
        if name:
            entry["name_variants"].add(name)
        phone = contact.get("phone", "").strip()
        if phone:
            entry["numbers"].add(phone)
            entry["numbers"].add(number_key)
        email = contact.get("email", "").strip().lower()
        if email:
            entry["emails"].add(email)

    return [
        {"name": item["name"],
         "name_variants": sorted(item["name_variants"]),
         "numbers": sorted(item["numbers"]),
         "emails": sorted(item["emails"])}
        for item in merged.values()
    ]

## Stage 3 – Small example and hand verification

In [6]:
contacts = [
    {"name": "Dr. Ravi", "phone": "+91 98400 12345", "email": "RAVI@example.com"},
    {"name": "ravi kumar", "phone": "9840012345", "email": "ravi.work@example.com"},
    {"name": "Meena", "phone": "09876543210", "email": "meena@example.com"},
    {"name": "MEENA", "phone": "+91 98765 43210", "email": ""},
    {"name": "Arun", "phone": "", "email": "arun@example.com"},
    {"name": "Arun", "phone": "", "email": "arun2@example.com"},
    {"name": "Ravi Office", "phone": "044-2345-6789", "email": ""}
]

result = merge_contacts(contacts)
for contact in result:
    print(contact)

assert len(result) == 5
ravi = next(item for item in result if "Dr. Ravi" in item["name_variants"])
assert len(ravi["emails"]) == 2
assert "9840012345" in ravi["numbers"]
assert len([x for x in result if x["name"] == "Arun"]) == 2
print("\nHand-check passed: 7 input records became 5 output contacts.")

{'name': 'Dr. Ravi', 'name_variants': ['Dr. Ravi', 'ravi kumar'], 'numbers': ['+91 98400 12345', '9840012345'], 'emails': ['ravi.work@example.com', 'ravi@example.com']}
{'name': 'Meena', 'name_variants': ['MEENA', 'Meena'], 'numbers': ['+91 98765 43210', '09876543210', '9876543210'], 'emails': ['meena@example.com']}
{'name': 'Arun', 'name_variants': ['Arun'], 'numbers': [], 'emails': ['arun@example.com']}
{'name': 'Arun', 'name_variants': ['Arun'], 'numbers': [], 'emails': ['arun2@example.com']}
{'name': 'Ravi Office', 'name_variants': ['Ravi Office'], 'numbers': ['044-2345-6789', '4423456789'], 'emails': []}

Hand-check passed: 7 input records became 5 output contacts.


## Stage 3 – Measure 1,000 and 100,000 contacts
The data generator creates duplicate pairs with two different formats of the same phone number. Each method is timed three times and the median is reported.

In [7]:
import time
import statistics


def make_contacts(n):
    records = []
    for i in range(n):
        person = i // 2
        digits = f"9{person % 10}{(person // 10) % 100000000:08d}"
        phone = digits if i % 2 == 0 else f"+91 {digits[:5]} {digits[5:]}"
        records.append({
            "name": f"Person {person}" if i % 2 == 0 else f"person {person}",
            "phone": phone,
            "email": f"p{person}@example.com" if i % 3 else ""
        })
    return records


benchmark = []
for size in (1_000, 100_000):
    data = make_contacts(size)
    times = []
    merged = None
    for _ in range(3):
        start = time.perf_counter()
        merged = merge_contacts(data)
        times.append(time.perf_counter() - start)
    benchmark.append({
        "Input contacts": size,
        "Merged contacts": len(merged),
        "Median time (seconds)": statistics.median(times)
    })

for row in benchmark:
    print(row)

{'Input contacts': 1000, 'Merged contacts': 500, 'Median time (seconds)': 0.004197505999854911}
{'Input contacts': 100000, 'Merged contacts': 50000, 'Median time (seconds)': 0.9679120130003867}


## Alternative: pairwise comparison
A pairwise design compares each contact with every later contact and merges records whose normalized phone numbers match. This requires O(n²) comparisons. The next cell measures it at 1,000 records only; 100,000 records would require billions of comparisons, so that run is intentionally omitted.

In [8]:
def pairwise_duplicate_keys(data):
    duplicate_keys = set()
    for i in range(len(data)):
        first = normalize_phone(data[i].get("phone", ""))
        if not first:
            continue
        for j in range(i + 1, len(data)):
            second = normalize_phone(data[j].get("phone", ""))
            if first == second:
                duplicate_keys.add(first)
    return duplicate_keys


data_1000 = make_contacts(1_000)
start = time.perf_counter()
pairwise_keys = pairwise_duplicate_keys(data_1000)
pairwise_time = time.perf_counter() - start

start = time.perf_counter()
dictionary_output = merge_contacts(data_1000)
dictionary_time = time.perf_counter() - start

print("Pairwise comparison (1,000):", round(pairwise_time, 6), "seconds")
print("Dictionary merge (1,000):", round(dictionary_time, 6), "seconds")
print("Duplicate phone keys found:", len(pairwise_keys))

Pairwise comparison (1,000): 0.81502 seconds
Dictionary merge (1,000): 0.004518 seconds
Duplicate phone keys found: 500


## Stage 4 – Defend the design
- Dictionary grouping is better than comparing every pair because average dictionary insertion and lookup take O(1), giving approximately O(n) total grouping time. Pairwise comparison takes O(n²).
- Two people with the same name but different normalized phone numbers remain separate. The program does not use name similarity alone to merge contacts.
- **Limitation:** a person who changes their phone number cannot be identified as the same person unless the records share another verified identifier. A shared family number can also cause two people to be merged.

## Alternative concept and complexity
- **Dictionary method:** O(n) average time and O(n) extra space for dictionary entries and sets.
- **Pairwise method:** O(n²) time and O(n) space for the resulting groups.
- At 1,000 contacts, the notebook measures both methods. At 100,000 contacts, the dictionary method is measured; pairwise comparison is omitted because its quadratic workload is impractical.

## Conclusion
The dictionary approach is suitable for large contact lists because it avoids checking every pair. The small example verifies duplicate merging, different number formats, missing fields and preservation of e-mail addresses. The rule is efficient but depends on phone numbers being reliable identifiers.